In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/review_sentiment.csv"
df = pd.read_csv(path)

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
df.head()

,review,sentiment
0,"$25,000 Pyramid Clues: Deep Blue Sea. Tremors....",negative
1,0.5/10. This movie has absolutely nothing good...,negative
2,"0*'s Christian Slater, Tara Reid, Stephen Dorf...",negative
3,"102 Dalmatians (2000, Dir. Kevin Lima) <br /><...",negative
4,102 DALMATIANS [Walt Disney]: I wasn't a fan o...,negative


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
POSITIVE,963799
NEGATIVE,481164
negative,74698
positive,74698
Neutral,55770
1,25000
0,25000
Positive,24582
Negative,20341


In [ ]:
mapping = {
    "POSITIVE": "Positive",
    "Positive": "Positive",
    "positive": "Positive",
    "pos": "Positive",
    1: "Positive",
    "1": "Positive",

    "NEGATIVE": "Negative",
    "Negative": "Negative",
    "negative": "Negative",
    "neg": "Negative",
    0: "Negative",
    "0": "Negative",

    "Neutral": "Neutral",
    "NEUTRAL": "Neutral",
    "neutral": "Neutral"
}

df["sentiment"] = df["sentiment"].replace(mapping)

In [ ]:
def check_values(df):
  return df.isnull().sum(), df.duplicated().sum()


missing_values,duplicated = check_values(df)

In [ ]:
duplicated

np.int64(230039)

In [ ]:
missing_values

,0
review,69225
sentiment,0


In [ ]:
df.shape

(1770052, 2)

In [ ]:
df = df.dropna(subset=["review"])

df.shape

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"http\S+|www\S+", " ", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text.strip()

df["review"] = df["review"].apply(clean_text)

df = df[df["review"].str.strip() != ""]

df = df[df["review"].str.split().str.len() >= 4]

df = df.drop_duplicates(subset=["review"])

df.shape

(1442896, 2)

In [ ]:
df["review"] = df["review"].apply(clean_text)

In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
Positive,941980
Negative,479373
Neutral,21543


In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

df['sentiment'] = encoder.fit_transform(df["sentiment"])

In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
2,941980
0,479373
1,21543


In [ ]:
print("Duplicate Rows :", df.duplicated(subset=["review", "sentiment"]).sum())

df = df.drop_duplicates(subset=["review", "sentiment"], keep="first")

print("Duplicate Rows After Remove :", df.duplicated(subset=["review", "sentiment"]).sum())

Duplicate Rows : 0
Duplicate Rows After Remove : 0


In [ ]:
df = df[df["review"].str.len() >= 15]

In [ ]:
print(df.shape)

(1442331, 2)


In [ ]:
df = df.sample(n=1442331, random_state=42)

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
train_df = pd.DataFrame({
    "review": X_train,
    "sentiment": y_train
})


print(train_df["sentiment"].value_counts())

sentiment
2    753460
0    383390
1     17014
Name: count, dtype: int64


In [ ]:
y_train.value_counts()

,count
sentiment,
2,753460
0,383390
1,17014


In [ ]:
from sklearn.utils import resample

class_0 = train_df[train_df["sentiment"] == 0]
class_1 = train_df[train_df["sentiment"] == 1]
class_2 = train_df[train_df["sentiment"] == 2]


class_0 = resample(
    class_0,
    replace=False,
    n_samples=380000,
    random_state=42
)

class_1 = resample(
    class_1,
    replace=True,
    n_samples=200000,
    random_state=42
)

class_2 = resample(
    class_2,
    replace=False,
    n_samples=450000,
    random_state=42
)

train_df = pd.concat([class_0, class_1, class_2])
train_df = train_df.sample(frac=1, random_state=42)

X_train = train_df["review"]
y_train = train_df["sentiment"]

In [ ]:
X_train.shape

(1030000,)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [ ]:
tokenizer = Tokenizer(
    num_words=35000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=200,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=200,
    padding="post",
    truncating="post"
)

In [ ]:
model = Sequential()

model.add(Embedding(
    input_dim=35000,
    output_dim=128,
    input_length=200
))

model.add(Bidirectional(LSTM(128)))

model.add(Dropout(0.4))

model.add(Dense(128, activation="relu"))

model.add(Dropout(0.3))

model.add(Dense(3, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
optimizer = Adam(learning_rate=0.0004)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 181s 56ms/step - accuracy: 0.8727 - loss: 0.3133 - val_accuracy: 0.8628 - val_loss: 0.3327
Epoch 2/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 176s 55ms/step - accuracy: 0.8885 - loss: 0.2780 - val_accuracy: 0.8639 - val_loss: 0.3302
Epoch 3/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 175s 54ms/step - accuracy: 0.9009 - loss: 0.2493 - val_accuracy: 0.8664 - val_loss: 0.3317
Epoch 4/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 175s 54ms/step - accuracy: 0.9124 - loss: 0.2241 - val_accuracy: 0.8626 - val_loss: 0.3500
Epoch 5/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 175s 54ms/step - accuracy: 0.9216 - loss: 0.2019 - val_accuracy: 0.8604 - val_loss: 0.3866
Epoch 6/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 175s 54ms/step - accuracy: 0.9304 - loss: 0.1811 - val_accuracy: 0.8610 - val_loss: 0.3928
Epoch 7/20
3219/3219 ━━━━━━━━━━━━━━━━━━━━ 174s 54ms/step - accuracy: 0.9379 - loss: 0.1631 - val_accuracy: 0.8582 - val_loss: 0.4440


In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

9015/9015 ━━━━━━━━━━━━━━━━━━━━ 81s 9ms/step - accuracy: 0.8312 - loss: 0.4094
Test Loss: 0.4094292223453522
Test Accuracy: 0.8312042355537415


In [ ]:
y_pred = model.predict(X_test_pad)

y_pred = np.argmax(y_pred, axis=1)

NameError: name 'model' is not defined

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.86      0.78     95848
           1       0.43      0.77      0.55      4253
           2       0.93      0.82      0.87    188366

    accuracy                           0.83    288467
   macro avg       0.69      0.81      0.73    288467
weighted avg       0.85      0.83      0.84    288467



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 81966   1917  11965]
 [   613   3266    374]
 [ 31400   2423 154543]]


In [ ]:
model.save("sentiment_bilstm.keras")

In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)